# Tennis XML Demo

Use the dropdown to pick a sample clip from `data/samples/`.

The XML rerenders automatically when you change the selection. Use **Force rerun** only when you want to rebuild the cached tracking artifacts for the current video.

This notebook exports a compact frame-and-rally XML structure:
- `match`
- `court`
- `frame` elements with `event="none"` outside rallies
- `rally` blocks with `frame` elements for `rally_start`, `contact`, and `rally_end`

The XML uses image-space coordinates. The rally and event tags come from the canonical `match_events.json` layer.

Each selected video gets its own cached tracking JSON, event JSON, and XML file inside `out/xml/`, so switching videos always shows the matching XML.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from tennis_tracker.xml_export import (
    ensure_match_tracking_xml_for_video,
    summarize_match_tracking_xml,
)

SAMPLES_DIR = ROOT / "data" / "samples"
SAMPLE_VIDEOS = sorted(SAMPLES_DIR.glob("*.mp4"))
assert SAMPLE_VIDEOS, f"No sample videos found in {SAMPLES_DIR}"

DEFAULT_VIDEO = next((p for p in SAMPLE_VIDEOS if p.name == "broadcast_2s.mp4"), SAMPLE_VIDEOS[0])
WEIGHTS_PATH = ROOT / "models" / "tracknet_weights.pth"
XML_OUT_DIR = ROOT / "out" / "xml"
XML_OUT_DIR.mkdir(parents=True, exist_ok=True)

assert WEIGHTS_PATH.exists(), f"Missing TrackNet weights: {WEIGHTS_PATH}"

selector = widgets.Dropdown(
    options=[(video.name, str(video)) for video in SAMPLE_VIDEOS],
    value=str(DEFAULT_VIDEO),
    description="Video:",
    layout=widgets.Layout(width="430px"),
)
force_rerun = widgets.Checkbox(value=False, description="Force rerun")
render_button = widgets.Button(description="Render XML", button_style="primary")
ui_output = widgets.Output()
def render_match_xml(video_path: Path, *, force: bool = False) -> None:
    ui_output.clear_output(wait=True)
    with ui_output:
        print(f"Rendering {video_path.name}...")

    artifacts = ensure_match_tracking_xml_for_video(
        video_path=video_path,
        output_dir=XML_OUT_DIR,
        weights_path=WEIGHTS_PATH,
        force=force,
    )
    xml_text = artifacts["xml"].read_text()

    summary = {
        "selected_video": video_path.name,
        "tracking_json": str(artifacts["tracking_json"].relative_to(ROOT)),
        "events_json": str(artifacts["events_json"].relative_to(ROOT)),
        "saved_xml": str(artifacts["xml"].relative_to(ROOT)),
        **summarize_match_tracking_xml(artifacts["xml"]),
    }

    xml_view = widgets.Textarea(
        value=xml_text,
        layout=widgets.Layout(width="100%", height="520px"),
        disabled=True,
    )

    ui_output.clear_output(wait=True)
    with ui_output:
        print(summary)
        display(xml_view)


In [2]:
def render_selected(*, force: bool = False) -> None:
    render_match_xml(Path(selector.value), force=force)


def on_render_clicked(_: widgets.Button) -> None:
    render_selected(force=force_rerun.value)


def on_video_change(change: dict) -> None:
    if change.get("name") != "value":
        return
    if change.get("new") == change.get("old"):
        return
    render_selected(force=False)


render_button.on_click(on_render_clicked)
selector.observe(on_video_change, names="value")
controls = widgets.HBox([selector, force_rerun, render_button])
display(widgets.VBox([controls, ui_output]))
render_selected(force=False)
